# Toolkit de funções de ML

Funções reutilizáveis que uso em projetos de modelagem de risco/propensão: EDA, outliers, **IV com optimal binning**,
lift, seleção de variáveis, pré-processamento, avaliação com **KS**, **análise por faixas e pontos de corte** para o
negócio, registro no **MLflow / Unity Catalog** e **PSI** para monitoramento.

As versões de produção (testadas e usadas pelos jobs deste projeto) estão em `src/credit_scoring/`:

| Função aqui | Versão de produção |
|---|---|
| `analise_binning2`, `plot_lift` | `binning.py` |
| análise por faixas, pontos de corte, KS | `evaluation.py` (`tabela_faixas`, `tabela_cortes`, `ks_statistic`) |
| PSI | `monitoring.py` |
| registro no MLflow / Unity Catalog | `serving.py` (`log_model`) |

Os nomes de dados (`df_train`, `X1`, `target`...) são genéricos: o notebook é um catálogo de referência, não é
executado de ponta a ponta.

### Funções de plotar gráficos de distribuição de variáveis e sua relação com o target

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


def plot_distribuicao_numerica(
    df, features, bins=30, kde=True, n_cols=3, figsize_unit=(5, 4)
):
    """Plota o histograma com curva de densidade (KDE) para variáveis contínuas.

    Parâmetros:
    -----------
    df : pd.DataFrame
        Conjunto de dados.
    features : str ou list
        Nome de uma coluna (ex: 'MonthlyCharges') ou lista (ex: ['tenure',
        'MonthlyCharges']).
    bins : int, padrão 30
        Número de faixas/barras do histograma.
    kde : bool, padrão True
        Se True, traça a linha suave de densidade sobre o histograma.
    n_cols : int, padrão 3
        Número máximo de colunas no grid de gráficos.
    figsize_unit : tuple, padrão (5, 4)
        Tamanho (largura, altura) de cada subplot individual.
    """
    # 1. Permite passar uma única feature ou uma lista
    if isinstance(features, str):
        features = [features]

    total_vars = len(features)
    n_cols = min(n_cols, total_vars)
    n_rows = math.ceil(total_vars / n_cols)

    # 2. Cria figura e eixos
    figsize = (n_cols * figsize_unit[0], n_rows * figsize_unit[1])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = np.array(axes).flatten()

    # 3. Desenha cada histograma
    for i, col in enumerate(features):
        ax = axes[i]

        sns.histplot(data=df, x=col, kde=kde, bins=bins, ax=ax)

        ax.set_title(f"Distribuição de {col}", fontsize=12, fontweight="bold")
        ax.set_ylabel("Frequência")
        ax.set_xlabel(col)

        # Remove bordas desnecessárias (topo e direita)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # 4. Remove eixos vazios caso o grid seja maior que o número de variáveis
    for j in range(total_vars, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()
    
# PARA UTILIZAR:
# 1. Apenas uma variável
plot_distribuicao_numerica(df_train, "X1")

# 2. Lista de variáveis
colunas_continuas = ["tenure", "MonthlyCharges", "TotalCharges"]
plot_distribuicao_numerica(df_train, colunas_continuas, bins=25, n_cols=3)

# 3. Desativando a linha KDE (deixando apenas as barras):
plot_distribuicao_numerica(df_train, "tenure", kde=False)

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np


def plot_taxa_target(df, features, target, n_cols=3, figsize_unit=(6, 4)):
    """Plota a taxa média da variável target para cada categoria das features.

    Calcula a média do target por categoria (ex: taxa de churn, taxa de conversão)
    e exibe em barras horizontais com os rótulos de porcentagem na ponta.
    """
    # 1. Permite passar uma única feature ou uma lista
    if isinstance(features, str):
        features = [features]

    total_vars = len(features)
    n_cols = min(n_cols, total_vars)
    n_rows = math.ceil(total_vars / n_cols)

    # 2. Cria a figura e os eixos
    figsize = (n_cols * figsize_unit[0], n_rows * figsize_unit[1])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = np.array(axes).flatten()

    # 3. Itera sobre cada feature e desenha as barras com os valores
    for i, col in enumerate(features):
        ax = axes[i]

        # Calcula a taxa média do target por categoria (em %)
        dados = (
            df.groupby(col)[target]
            .mean()
            .mul(100)
            .sort_values(ascending=True)
            .reset_index()
        )

        # Gráfico de barras horizontais
        barras = ax.barh(y=dados[col].astype(str), width=dados[target])

        # Adiciona o rótulo com a porcentagem ao final de cada barra
        ax.bar_label(barras, fmt="%.1f%%", padding=3, fontsize=10)

        # Ajustes visuais limpos
        ax.set_title(f"{col} vs {target}", fontsize=12, fontweight="bold")
        ax.set_xlabel(f"Taxa média de {target} (%)")
        ax.set_xlim(
            0, max(dados[target].max() * 1.15, 10)
        )  # Espaço extra para o texto caber

        # Remove bordas desnecessárias (topo e direita)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # 4. Remove eixos sobressalentes do grid
    for j in range(total_vars, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()
    
    
# PARA UTILIZAR:
# 1. Apenas uma variável
plot_taxa_target(df_train, features="X1", target="Y")

# 2. Lista de variáveis
colunas = ["X1", "X2", "X3"]
plot_taxa_target(df_train, features=colunas, target="Y", n_cols=2)

### Funções para identificação de outliers

In [ ]:
def plot_histogram_outliers(variable):
  """
  Gera um histograma com linha de densidade (KDE) e um boxplot
  para visualizar a distribuição e os outliers de uma variável.

  Args:
      variable (str): O nome da coluna (variável) a ser plotada.
  """
  fig, axes = plt.subplots(1, 2, figsize=(17, 2))

  # Gera o histograma com a linha de densidade (KDE)
  sns.histplot(x=variable, data=df_train, ax=axes[0], kde=True)

  # Gera o boxplot
  sns.boxplot(x=variable, data=df_train, ax=axes[1])

  plt.show()

In [ ]:
import pandas as pd


def verificar_outliers(df, features):
    """Calcula a quantidade e percentual de outliers usando o método do IQR (Intervalo Interquartil).

    Aceita uma única coluna (str) ou lista de colunas.
    """
    if isinstance(features, str):
        features = [features]

    relatorio = []

    for col in features:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        limite_inferior = q1 - 1.5 * iqr
        limite_superior = q3 + 1.5 * iqr

        # Filtra os outliers
        mascara = (df[col] < limite_inferior) | (df[col] > limite_superior)
        qtd_outliers = mascara.sum()
        pct_outliers = (qtd_outliers / len(df)) * 100

        relatorio.append(
            {
                "coluna": col,
                "outliers": qtd_outliers,
                "pct (%)": round(pct_outliers, 2),
                "limite_inferior": round(limite_inferior, 2),
                "limite_superior": round(limite_superior, 2),
            }
        )

    # Retorna uma tabela limpa e fácil de ler no notebook
    return pd.DataFrame(relatorio)

### Painel de Análise Bivariada

Para cada variável, a tabela do optimal binning vira um painel com as colunas abaixo.

* **1. Optimal binning:** a variável contínua é discretizada em faixas que maximizam o poder preditivo, com taxa de
  evento monotônica e tamanho mínimo por faixa (robustez estatística, sem faixas ruidosas).
* **2. Distribuição e taxa de evento:** onde se concentra a base (% Total) e como a taxa de evento evolui ao longo
  das faixas, comprovando a ordenação do risco.

### 3. ODDS (razão de chance)
Chance de o evento acontecer em relação a não acontecer dentro da faixa:
$$\text{ODDS} = \frac{\text{Eventos}}{\text{Não-Eventos}} = \frac{\text{Taxa Evento}}{1 - \text{Taxa Evento}}$$
Exemplo: ODDS de 1,83 → para cada 100 casos sem o evento, 183 com o evento.

### 4. IV (Information Value)
Poder preditivo consolidado da variável — *"vale a pena manter essa informação no modelo?"*:
$$\text{IV} = \sum_{i} \left( \%\,\text{Evento}_i - \%\,\text{Não-Evento}_i \right) \times \ln\left( \frac{\%\,\text{Evento}_i}{\%\,\text{Não-Evento}_i} \right)$$
onde $\%\,\text{Evento}_i$ é a fração de todos os eventos da base que caiu na faixa $i$.

| IV | Leitura |
|---|---|
| < 0,02 | não separa |
| 0,02 – 0,10 | fraco |
| 0,10 – 0,30 | médio |
| 0,30 – 0,50 | forte |
| > 0,50 | investigar (possível leakage) |

### 5. Lift
Ganho relativo à taxa média da base — orienta cortes, alocação de recursos e segmentação:
$$\text{Lift} = \frac{\text{Taxa Evento da faixa}}{\text{Taxa Evento média da base}}$$
Exemplo: com taxa média de 25%, uma faixa com lift 1,57 entrega um resultado 57% melhor que escolher ao acaso.

> Versão de produção (testada): `src/credit_scoring/binning.py` — `analise_binning`, `bivariate_table`, `plot_lift`.

In [ ]:
# QUEBRAS DO OPTIMAL BINNING PARA GRÁFICO DE BIVARIADAS

import pandas as pd
import numpy as np
from optbinning import OptimalBinning

# Ler tabela salva
tabela_fonte = "<catalogo>.<schema>.<tabela>"
df_bi = spark.table(tabela_fonte).toPandas()

# Manter apenas registros com target definido
df_bi = df_bi[df_bi['target'].notna()].reset_index(drop=True)

# Lista de features
features = [
    'X1',
    'X2',
    'X3',
]

# Target
r_col = 'target'

# Parametros do optimal binning 
filtro = ['Totals', 'Special', 'Missing']
tipo = 'numerical'
sol = 'cp'
tam = 0.01
n_bins = 5
modelo = 'quantile'

# Cortes manuais para variaveis com distribuicao concentrada em zero
# (quantile prebinning falha quando >60% dos valores sao iguais)
user_splits_custom = {
    'XO1': [0.5, 2.5, 5.5],
    'XO2': [0.001, 0.05, 0.5]
}

# Variaveis binarias (0/1) -- tratar como categoricas
vars_binarias = ['XF1', 'XF2']

# Funcao analise_binning2 
def analise_binning2(base, p_cols, r_col, filtro, tipo, sol, tam, n_bins, modelo):
    bivariadas = {}
    df2 = pd.DataFrame()

    if r_col in p_cols:
        p_cols.remove(r_col)

    for i in p_cols:
        print(i)
        # Variaveis binarias -- tratar como categoricas
        if i in vars_binarias:
            optb = OptimalBinning(
                name=i, dtype='categorical', solver=sol,
                min_prebin_size=tam, min_bin_size=tam
            )
        # Variaveis com cortes manuais (concentradas em zero)
        elif i in user_splits_custom:
            optb = OptimalBinning(
                name=i, dtype=tipo, solver=sol,
                min_prebin_size=tam, min_bin_size=tam,
                user_splits=user_splits_custom[i],
                user_splits_fixed=[True] * len(user_splits_custom[i])
            )
        # Demais variaveis -- quantile prebinning padrao
        else:
            optb = OptimalBinning(
                name=i, dtype=tipo, solver=sol,
                min_prebin_size=tam, min_bin_size=tam,
                max_n_prebins=n_bins, max_n_bins=n_bins,
                prebinning_method=modelo
            )
        optb.fit(base[i].values, base[r_col])
        binning_table = optb.binning_table
        df = pd.DataFrame(binning_table.build())
        df = df[~df['Bin'].isin(filtro)]
        df.rename(columns={
            'Bin': 'Categoria',
            'Count': 'Qtde',
            'Count (%)': '% Total',
            'Non-event': 'Nao Evento',
            'Event': 'Evento',
            'Event rate': 'Taxa Evento'
        }, inplace=True)
        # Garantir que Categoria e sempre string (categoricas retornam arrays)
        df['Categoria'] = df['Categoria'].astype(str)
        df['% Nao evento'] = df['Nao Evento'] / df['Nao Evento'].sum()
        df['% Evento'] = df['Evento'] / df['Evento'].sum()
        df['Odds'] = df['% Evento'] / df['% Nao evento']
        df['Variavel'] = i
        df['Lift'] = df['Taxa Evento'] / (df['Evento'].sum() / df['Qtde'].sum())
        df = df[['Variavel', 'Categoria', 'Evento', 'Nao Evento', 'Qtde',
                 'Taxa Evento', 'Odds', 'IV', '% Total', 'Lift', '% Evento', '% Nao evento']]
        bivariadas[i] = df
        df2 = pd.concat([df2, df])

    return bivariadas, df2

# Executar optimal binning
bivariadas, df2 = analise_binning2(df_bi, features.copy(), r_col, filtro, tipo, sol, tam, n_bins, modelo)

### Lift

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_lift(bivariadas, variavel):
    df = bivariadas[variavel]
    df = df.loc[df.index != 'Totals']
    df = df.reset_index(drop=True)

    x = np.arange(len(df))

    fig, ax1 = plt.subplots(figsize=(10, 5))
    bar_width = 0.35

    ax1.bar(x - bar_width/2, df['% Evento']*100, bar_width, label='% Evento')
    ax1.bar(x + bar_width/2, df['% Total']*100, bar_width, label='% Não Evento')

    ax1.set_xlabel('Categoria', fontsize=16)
    ax1.set_ylabel('% Evento e % Total', fontsize=16)
    ax1.set_xticks(x)
    ax1.set_xticklabels(df['Categoria'])

    ax2 = ax1.twinx()
    ax2.plot(x, df['Lift'], 'r-', label='Lift')

    for i, v in enumerate(df['Lift']):
        ax2.text(i, v+0.02, f'{v:.2f}', ha='center')

    ax2.set_ylabel('Lift', fontsize=16)
    ax2.axhline(1, color='black', linestyle='--')

    plt.title(f'Binning - {variavel}')
    plt.tight_layout()
    plt.show()

# Seleção de variáveis

In [ ]:
#CALCULO DO IV e Gráficos do LIFT
%pip install optbinning

import pandas as pd
from sklearn.tree import DecisionTreeClassifier,export_graphviz, plot_tree, export_text
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
import plotly.express as px
from sklearn.preprocessing import OneHotEncoder

import numpy as np
from optbinning import OptimalBinning
from plotly.subplots import make_subplots

import plotly.graph_objects as go

def grafico_binning_plot(df, variavel):

    df = df[df['Variável']==variavel]
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Extract traces from the bar plot and add them to the subplot
    bar_fig = px.bar( df[df['Categoria'] != '']
                     , x='Categoria'
                     , y=['% Evento', '% Total']
                     , barmode='group'
                     , color_discrete_sequence=px.colors.qualitative.Safe
                     )

    for trace in bar_fig['data']:
        fig.add_trace(trace, secondary_y=False)

    fig.add_trace(go.Scatter(x=df[df['Categoria'] != '']['Categoria'], y=df[df['Categoria'] != '']['Lift'],
                    mode='lines+markers+text',
                    name='Lift', text=list((df[df['Categoria'] != '']['Lift']).round(3).astype(str)),
                    textposition="top center",
                    line=dict(color='black'))
              , secondary_y=True)

    fig.update_layout(plot_bgcolor='white'
                      , paper_bgcolor='white'
                      , font=dict(size=16)
                      , title='OptBinning - variável %s' % variavel
                      , bargap=0.2
                      , bargroupgap=0.1
                      )

    fig.update_layout(xaxis=dict(tickmode='linear'))
    # Centralizando o título
    fig.update_layout(title={'x':0.5})


    fig.update_layout(width=1350,  # Largura do gráfico
                      height=550  # Altura do gráfico
                      )
    return fig, df

def analise_binning2(base, p_cols, r_col, filtro, tipo, sol, tam, n_bins, modelo):

    bivariadas = {}
    df2 = pd.DataFrame()

    if r_col in p_cols:
        p_cols.remove(r_col)

    #plots=[]
    for i in p_cols:
        print(i)
        #basepandas[i].fillna(0, inplace=True)
        optb = OptimalBinning(name=i, dtype=tipo, solver=sol, min_prebin_size = tam, min_bin_size = tam, max_n_prebins = n_bins, max_n_bins= n_bins,  prebinning_method = modelo)
        optb.fit(base[i].values, base[r_col])
        optb.splits
        binning_table = optb.binning_table
        df= pd.DataFrame(binning_table.build())
        df=df[~df['Bin'].isin(filtro)]
        df.rename(columns={'Bin': 'Categoria', 'Count': 'Qtde', 'Count (%)': '% Total','Non-event': 'Não Evento','Event': 'Evento','Event rate': 'Taxa Evento'}, inplace=True)
        df["% Não evento"] = df["Não Evento"] / df["Não Evento"].sum()
        df["% Evento"] = df["Evento"] / df["Evento"].sum()
        df["Odds"] = df["% Evento"] / df["% Não evento"]
        df['Variável']= i
        df["Lift"] = df['Taxa Evento']/(df['Evento'].sum()/df['Qtde'].sum())
        df = df[['Variável', 'Categoria', 'Evento','Não Evento', 'Qtde','Taxa Evento','Odds','IV','% Total','Lift','% Evento','% Não evento']]
        bivariadas[i]=df
        df2 = pd.concat([df2, df])

    return bivariadas, df2

def iv_bao(analise_iv, corte, figura):

    iv_bao = []
    conta = 0

    iv_bao2 = {} 

    for i in figura:
        a = analise_iv[i]['IV'].loc['Totals']
        if ((a > corte)): #& (a < 0.50)):
            iv_bao.append(i)
            #print(conta, i, a, 'BAO')
            iv_bao2[i] = a   

        conta = conta + 1 

    iv_bao2 = (pd.DataFrame([iv_bao2]).T).sort_values(by=0, ascending=False).rename(columns={0: 'IV'})

    #print('TOTAL DE VARIÁVEIS BOAS', len(iv_bao2))

    return iv_bao2

def analise_odds_bin_opt(dfs, variaveis, analise_iv_df, dt_ref, target):

    resultado = pd.DataFrame()
    resultado2 = pd.DataFrame()
    cont = 0

    for i in variaveis:

        df_variavel = analise_iv_df[(analise_iv_df['Variável'] == i) & (analise_iv_df['Categoria'] != '')]
        categoria_lista = df_variavel['Categoria'].tolist()

        perc = [-float('inf')] + list(map(float, dict.fromkeys(','.join(categoria_lista).replace(' ', '').replace('(-inf', '').replace('inf)', '').replace(')', '').replace('[', '').split(',')[1:-1]))) + [float('inf')]

        for df in dfs:

            df["Classe"] = pd.DataFrame(pd.cut(np.array(df[i]), perc))

            odds_df = df.groupby(['Classe', target]).agg(P=pd.NamedAgg(column=i, aggfunc="count")).reset_index()
            odds_df_pivot = odds_df.pivot(index='Classe', columns=target, values='P').fillna(0)
            odds_df_pivot['ODDs'] = (odds_df_pivot[1] / odds_df_pivot[0]).round(3)
            odds_df_pivot['WOE'] = np.log(odds_df_pivot['ODDs']).round(3)
            odds_df_pivot.reset_index(inplace=True)
            odds_df_pivot['Classe'] = odds_df_pivot['Classe'].astype(str)
            odds_df_pivot['dt_ref'] = str(cont) + ". " + dt_ref[cont]

            # Calculate standard deviation and range of WOE
            woe_std = odds_df_pivot['WOE'].std().round(3)
            woe_range = (odds_df_pivot['WOE'].max() - odds_df_pivot['WOE'].min()).round(3)
            odds_df_pivot['WOE_STD'] = woe_std
            odds_df_pivot['WOE_RANGE'] = woe_range

            odds_df_pivot = odds_df_pivot.reset_index().astype(str)
            odds_df_pivot['Classe'] = odds_df_pivot['index'] + '. ' + odds_df_pivot['Classe']

            odds_df_pivot = odds_df_pivot.drop(columns = ['index'])

            resultado = pd.concat([resultado, odds_df_pivot])


            #if cont == 0:
            #    resultado2 = odds_df_pivot[['Classe', 'ODDs']]
            #else:


            cont += 1

    return resultado

import matplotlib.pyplot as plt
import numpy as np

def plot_lift(bivariadas, variavel):
    df = bivariadas[variavel]
    df = df.loc[df.index != 'Totals']
    df = df.reset_index(drop=True)

    x = np.arange(len(df))

    fig, ax1 = plt.subplots(figsize=(10, 5))
    bar_width = 0.35

    ax1.bar(x - bar_width/2, df['% Evento']*100, bar_width, label='% Evento')
    ax1.bar(x + bar_width/2, df['% Total']*100, bar_width, label='% Não Evento')

    ax1.set_xlabel('Categoria', fontsize=16)
    ax1.set_ylabel('% Evento e % Total', fontsize=16)
    ax1.set_xticks(x)
    ax1.set_xticklabels(df['Categoria'])

    ax2 = ax1.twinx()
    ax2.plot(x, df['Lift'], 'r-', label='Lift')

    for i, v in enumerate(df['Lift']):
        ax2.text(i, v+0.02, f'{v:.2f}', ha='center')

    ax2.set_ylabel('Lift', fontsize=16)
    ax2.axhline(1, color='black', linestyle='--')

    plt.title(f'Binning - {variavel}')
    plt.tight_layout()
    plt.show()

 

In [ ]:
# EXEMPLO DE GRÁFICO DE VISUALIZAÇÃO DE IVs COM PONTOS DE CORTE BAIXO, MÉDIO E ALTO

# CONSOLIDANDO OS RESULTADOS DE IV
iv_final = {
    "RevolvingUtilizationOfUnsecuredLines": 1.1134,
    "NumberOfTimes90DaysLate": 0.8774,
    "NumberOfTime30-59DaysPastDueNotWorse": 0.7552,
    "NumberOfTime60-89DaysPastDueNotWorse": 0.6000,
    "age": 0.2592,
    "MonthlyIncome": 0.0670,
    "NumberOfOpenCreditLinesAndLoans": 0.0669,
    "DebtRatio": 0.0253,
    "NumberOfDependents": 0.0250,
    "NumberRealEstateLoansOrLines": 0.0121,
    "flag_missing_income": 0.0077,
    "flag_missing_dependents": 0.0037
}

iv_final_df = pd.DataFrame.from_dict(
    iv_final, orient="index", columns=["IV"]
).sort_values("IV", ascending=True)

# Visualização
fig, ax = plt.subplots(figsize=(10, 7))
cores = ["#e74c3c" if v > 0.3 else "#f39c12" if v > 0.1
         else "#2ecc71" if v > 0.02 else "#95a5a6"
         for v in iv_final_df["IV"]]
ax.barh(iv_final_df.index, iv_final_df["IV"], color=cores)
ax.axvline(0.02, color="gray", linestyle="--", alpha=0.7, label="Fraco (0.02)")
ax.axvline(0.1, color="#f39c12", linestyle="--", alpha=0.7, label="Médio (0.1)")
ax.axvline(0.3, color="#e74c3c", linestyle="--", alpha=0.7, label="Forte (0.3)")
ax.set_title("Information Value — Ranking de Poder Preditivo", fontweight="bold")
ax.set_xlabel("IV")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/iv_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, SelectKBest, f_classif


def selecionar_features_classificacao(
    X_train, y_train, metodo="kbest", k=20, random_state=42
):
    """Seleciona as k melhores variáveis para problemas de classificação.

    Métodos:
    - 'kbest': Teste estatístico ANOVA (f_classif). Rápido, avalia cada coluna
    sozinha em relação às classes.
    - 'rfe': Eliminação Recursiva de Variáveis com Random Forest. Mais lento, mas avalia
    combinações entre as colunas.
    OBS: avaliação da relação entre variável e target fica para ser feito com SHAP.
    """
    if metodo == "kbest":
        seletor = SelectKBest(score_func=f_classif, k=k)
    elif metodo == "rfe":
        modelo = RandomForestClassifier(random_state=random_state)
        seletor = RFE(estimator=modelo, n_features_to_select=k)

    # Treina o seletor com os dados de treino
    seletor.fit(X_train, y_train)

    # Pega os nomes das colunas escolhidas
    colunas_escolhidas = X_train.columns[seletor.get_support()]

    # Filtra o DataFrame original mantendo índices e nomes das colunas
    X_train_reduzido = X_train[colunas_escolhidas]

    return X_train_reduzido, seletor

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, SelectKBest, f_regression


def selecionar_features_regressao(
    X_train, y_train, metodo="kbest", k=20, random_state=42
):
    """Seleciona as k melhores variáveis para problemas de regressão.

    Métodos:
    - 'kbest': Teste estatístico F (f_regression). Rápido, mede a correlação
    linear de cada coluna com o alvo contínuo.
    - 'rfe': Eliminação Recursiva de Variáveis com Random Forest Regressor. Mais lento, mas
    capta padrões não lineares entre colunas.
    OBS: avaliação da relação entre variável e target fica para ser feito com SHAP.
    """
    if metodo == "kbest":
        seletor = SelectKBest(score_func=f_regression, k=k)
    elif metodo == "rfe":
        modelo = RandomForestRegressor(random_state=random_state)
        seletor = RFE(estimator=modelo, n_features_to_select=k)

    # Treina o seletor com os dados de treino
    seletor.fit(X_train, y_train)

    # Pega os nomes das colunas escolhidas
    colunas_escolhidas = X_train.columns[seletor.get_support()]

    # Filtra o DataFrame original mantendo índices e nomes das colunas
    X_train_reduzido = X_train[colunas_escolhidas]

    return X_train_reduzido, seletor

# Pré-processamento

Escalonamento para Características Numéricas:

* utiliza-se o StandardScaler para normalizar os dados, garantindo que todas as características numéricas contribuam igualmente para o modelo.

Codificação para Características Categóricas:

 * Codificação One-Hot (One-Hot Encoding): Aplicada a características categóricas para evitar relações ordinais e permitir que o modelo interprete cada categoria de forma independente.

 * Codificação Ordinal (Ordinal Encoding): Aplicada à característica Contract (Contrato), pois ela possui uma ordem inerente (ex: Mês a mês, Um ano, Dois anos), preservando a sequência das categorias.

In [ ]:
# Colunas ordinais e a ordem desejada das categorias
ordinal_cols = ['coluna_ord_1']
ordinal_categories = [['baixa', 'media', 'alta']]  # lista de listas na ordem correta

# Colunas categóricas nominais (para One-Hot Encoding)
categorical_cols = [
    'coluna_cat_1',
    'coluna_cat_2',
    'coluna_cat_3'
]

# Colunas numéricas contínuas (para imputação por mediana + padronização)
numerical_cols = [
    'coluna_num_1',
    'coluna_num_2'
]

# Colunas binárias (já numéricas 0/1, apenas repassadas sem alteração)
binary_cols = [
    'coluna_bin_1'
]

In [ ]:
# ==============================================================================
# 2. TRANSFORMADORES E PRÉ-PROCESSADOR (Idêntico à sua estrutura)
# ==============================================================================

scaling_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

onehot_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

ordinal_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=ordinal_categories)),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        # Colunas Ordinais
        ('ordinal', ordinal_transformer, ordinal_cols),

        # Colunas Categóricas (One-Hot)
        ('categorical', onehot_transformer, categorical_cols),

        # Colunas Numéricas (Scaling)
        ('scaling', scaling_transformer, numerical_cols),

        # Colunas Binárias (Passthrough)
        ('binary', 'passthrough', binary_cols)
    ],
    remainder='drop'  # Descarta colunas não mapeadas
)

# Modelagem

In [ ]:
# ══════════════════════════════════════════════════════════════
# MODELO 1 — Regressão Logística
# Motivação: modelo linear interpretável, baseline inteligente
# Vantagem: coeficientes explicáveis para o comitê de crédito
# Limitação: assume relação linear entre features e target
# Requer: normalização das features (StandardScaler)
# ══════════════════════════════════════════════════════════════

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",  # trata desbalanceamento
        max_iter=1000,
        random_state=SEED,
        C=0.1                     # regularização L2 — controla multicolinearidade
    ))
])

lr_pipeline.fit(X_train, y_train)
resultado_lr = avaliar_modelo(
    "Regressão Logística", lr_pipeline,
    X_train, X_test, y_train, y_test
)

# ══════════════════════════════════════════════════════════════
# MODELO 2 — Random Forest
# Motivação: ensemble de árvores, captura não-linearidades
# Vantagem: robusto a outliers e multicolinearidade
# Limitação: menos interpretável, mais lento para treinar
# Não requer: normalização
# ══════════════════════════════════════════════════════════════

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,              # limitar profundidade — controla overfitting
    min_samples_leaf=50,      # mínimo de amostras por folha — regularização
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1                 # usar todos os cores disponíveis
)

rf_model.fit(X_train, y_train)
resultado_rf = avaliar_modelo(
    "Random Forest", rf_model,
    X_train, X_test, y_train, y_test
)

# ══════════════════════════════════════════════════════════════
# MODELO 3 — XGBoost
# Motivação: gradient boosting — estado da arte em dados tabulares
# Vantagem: performance superior, lida bem com desbalanceamento
# Limitação: mais hiperparâmetros, risco maior de overfitting
# ══════════════════════════════════════════════════════════════

# scale_pos_weight trata desbalanceamento no XGBoost
# fórmula: n_negativos / n_positivos
scale_pw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pw:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,              # árvores rasas — controla overfitting
    learning_rate=0.05,       # passo pequeno — mais robusto
    subsample=0.8,            # amostragem de linhas por árvore
    colsample_bytree=0.8,     # amostragem de features por árvore
    scale_pos_weight=scale_pw,
    random_state=SEED,
    eval_metric="auc",
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
resultado_xgb = avaliar_modelo(
    "XGBoost", xgb_model,
    X_train, X_test, y_train, y_test
)



### Técnicas de reamostragem

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, NearMiss
from imblearn.combine import SMOTEENN


def avaliar_resampling(preprocessor, X_train, y_train, cv=5, seed=123):
    """
    Executa a avaliação de modelos com técnicas de reamostragem
    via validação cruzada no conjunto de treino.
    """
    # 1. Definição dos classificadores
    classifiers = [
        ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=seed)),
        ('Random Forest', RandomForestClassifier(random_state=seed)),
        ('Gradient Boosting', GradientBoostingClassifier(random_state=seed)),
        ('XGBoost', XGBClassifier(eval_metric='logloss', random_state=seed)),
        ('LightGBM', LGBMClassifier(verbose=-1, random_state=seed))
    ]

    # 2. Definição dos métodos de reamostragem
    resamplers = [
        ('None', None),
        ('RandomOverSampler', RandomOverSampler(random_state=17)),
        ('RandomUnderSampler', RandomUnderSampler(random_state=17)),
        ('SMOTE', SMOTE(random_state=17)),
        ('NearMiss', NearMiss()),
        ('SMOTEENN', SMOTEENN(random_state=17))
    ]

    # 3. Estratégia de validação cruzada
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=seed)
    cv_results = []

    print("--- Avaliação de Modelos com Reamostragem (via Validação Cruzada) ---")

    # 4. Loop de avaliação
    for resample_name, resample in resamplers:
        for clf_name, clf in classifiers:
            print(f"Processando {clf_name} com {resample_name} ...")

            # Construção dos passos do pipeline
            steps = [('preprocessor', preprocessor)]
            if resample is not None:
                steps.append(('resampler', resample))
            steps.append(('classifier', clf))

            pipeline = Pipeline(steps=steps)

            try:
                cv_scores = cross_val_score(
                    pipeline,
                    X_train,
                    y_train,
                    cv=skf,
                    scoring='recall',
                    n_jobs=-1
                )

                cv_results.append({
                    'Model': clf_name,
                    'Resampler': resample_name,
                    'Train Recall Mean': cv_scores.mean(),
                    'Train Recall Std': cv_scores.std(),
                    'Train Recall Min': cv_scores.min()
                })

            except Exception as e:
                print(f"Erro processando {clf_name} com {resample_name}: {e}")

    # 5. Consolidação e ordenação do DataFrame
    df_results = pd.DataFrame(cv_results)
    df_results = df_results.sort_values(by='Train Recall Mean', ascending=False).reset_index(drop=True)

    return df_results

### Avaliação de modelos

In [ ]:
# CRIANDO UMA FUNÇÃO PARA AVALIAR OS MODELOS

def calcular_ks(y_true, y_proba):
    """KS — métrica padrão do mercado de crédito brasileiro"""
    from scipy.stats import ks_2samp
    scores_positivos = y_proba[y_true == 1]
    scores_negativos = y_proba[y_true == 0]
    ks_stat, _ = ks_2samp(scores_positivos, scores_negativos)
    return ks_stat

def avaliar_modelo(nome, modelo, X_train, X_test, y_train, y_test, cv=5):
    """
    Avalia um modelo com métricas completas + validação cruzada estratificada.
    Retorna dicionário com todas as métricas.
    """
    # Predições
    y_proba = modelo.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    # Métricas no teste
    auc = roc_auc_score(y_test, y_proba)
    ks = calcular_ks(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    # Validação cruzada estratificada no treino
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    cv_results = cross_validate(
        modelo, X_train, y_train,
        cv=skf,
        scoring="roc_auc",
        return_train_score=True
    )
    cv_auc_mean = cv_results["test_score"].mean()
    cv_auc_std = cv_results["test_score"].std()
    train_auc_mean = cv_results["train_score"].mean()

    print(f"\n{'='*50}")
    print(f"MODELO: {nome}")
    print(f"{'='*50}")
    print(f"ROC-AUC  (teste):       {auc:.4f}")
    print(f"KS       (teste):       {ks:.4f}")
    print(f"F1-Score (teste):       {f1:.4f}")
    print(f"Precision(teste):       {precision:.4f}")
    print(f"Recall   (teste):       {recall:.4f}")
    print(f"ROC-AUC  (CV treino):   {cv_auc_mean:.4f} ± {cv_auc_std:.4f}")
    print(f"ROC-AUC  (CV vs teste): {train_auc_mean:.4f} vs {auc:.4f}")
    print(f"Gap train/test: {abs(train_auc_mean - auc):.4f}", end=" ")
    print("Overfitting?" if abs(train_auc_mean - auc) > 0.05 else "Estável")

    return {
        "modelo": nome,
        "auc_teste": auc,
        "ks_teste": ks,
        "f1_teste": f1,
        "precision_teste": precision,
        "recall_teste": recall,
        "cv_auc_mean": cv_auc_mean,
        "cv_auc_std": cv_auc_std,
        "train_auc_mean": train_auc_mean
    }

In [ ]:
# COMPARAÇÃO DAS CURVAS ROC
fig, ax = plt.subplots(figsize=(10,7))

modelos_plot = [
    ("Regressao Logística", lr_pipeline, "#3498db"),
    ("Random Forest", rf_model, "#2ecc71"),
    ("XGBoost", xgb_model, "#e74c3c")
]

for nome, modelo, cor in modelos_plot:
    y_proba = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{nome} (AUC={auc:.4f})", color=cor, linewidth=2)
    
ax.plot([0,1], [0,1], "k--", linewidth=1, label="Baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves - Model Comparison", fontweight="bold")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../reports/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### Painel de análise bivariada para Gestores

In [ ]:
# ANALISE DE FAIXAS DE PROBABILIDADE DO EVENTO (TARGET = 1)

df_faixas = pd.DataFrame({
    'score': prob_rf,
    'classe_real': y_teste,
})

df_faixas = df_faixas.sort_values('score', ascending=False).reset_index(drop=True)

total_obs = len(df_faixas)
tamanho_faixa = total_obs // 40

numeros_linha = np.arange(total_obs)
array_faixas = (numeros_linha // tamanho_faixa).clip(0, 39) + 1
df_faixas['faixa'] = array_faixas

total_evento = (df_faixas['classe_real'].values == 1).sum()
total_nao_evento = (df_faixas['classe_real'].values == 0).sum()
taxa_media = total_evento / total_obs

resultados = []
for numero_faixa in range(1, 41):
    filtro_faixa = df_faixas['faixa'].values == numero_faixa
    grupo_faixa = df_faixas[filtro_faixa]

    qtd_total = len(grupo_faixa)
    qtd_evento = (grupo_faixa['classe_real'].values == 1).sum()
    qtd_nao_evento = (grupo_faixa['classe_real'].values == 0).sum()
    taxa_evento = qtd_evento / qtd_total
    lift = taxa_evento / taxa_media

    resultados.append({
        'faixa': numero_faixa,
        'qtd_evento': qtd_evento,
        'qtd_nao_evento': qtd_nao_evento,
        'qtd_total': qtd_total,
        'taxa_evento': round(taxa_evento, 4),
        'lift': round(lift, 2),
    })

df_resultado = pd.DataFrame(resultados)

df_resultado['evento_acum'] = df_resultado['qtd_evento'].cumsum()
df_resultado['nao_evento_acum'] = df_resultado['qtd_nao_evento'].cumsum()
df_resultado['pct_evento_acum'] = (df_resultado['evento_acum'] / total_evento).round(4)
df_resultado['pct_nao_evento_acum'] = (df_resultado['nao_evento_acum'] / total_nao_evento).round(4)
df_resultado['ks'] = (df_resultado['pct_evento_acum'] - df_resultado['pct_nao_evento_acum']).round(4)
df_resultado['precisao_acum'] = (df_resultado['evento_acum'] / (df_resultado['faixa'] * tamanho_faixa)).round(4)

print("Analise por faixas")
print(f"Total de linhas: {total_obs}")
print(f"Total de eventos: {total_evento}")
print(f"Taxa media de eventos: {taxa_media:.2%}")
print()

colunas_exibir = [
    'faixa',
    'qtd_evento',
    'qtd_nao_evento',
    'qtd_total',
    'taxa_evento',
    'pct_evento_acum',
    'pct_nao_evento_acum',
    'ks',
    'lift',
    'precisao_acum',
]

print(df_resultado[colunas_exibir].to_string(index=False))

idx_ks_max = df_resultado['ks'].values.argmax()
ks_max = float(df_resultado['ks'].values[idx_ks_max])
faixa_ks_max = int(df_resultado['faixa'].values[idx_ks_max])

filtro_faixa_1 = df_resultado['faixa'].values == 1
lift_faixa1 = float(df_resultado['lift'].values[filtro_faixa_1][0])
prec_top5 = float(df_resultado['taxa_evento'].values[filtro_faixa_1][0])
recall_top5 = float(df_resultado['pct_evento_acum'].values[filtro_faixa_1][0])

print()
print(f"KS maximo: {ks_max:.4f}")
print(f"Faixa do KS maximo: {faixa_ks_max}")
print(f"Lift da faixa 1: {lift_faixa1:.2f}x")
print(f"Precisao top 5%: {prec_top5:.4f}")
print(f"Recall top 5%: {recall_top5:.4f}")
 

In [ ]:
# THRESHOLDS E METRICAS DE AVALIAÇÃO 

pontos_de_corte = []
 
for numero_faixa in range(1, 41):
    filtro_faixa = df_faixas['faixa'].values == numero_faixa
    grupo_faixa = df_faixas[filtro_faixa]
    score_minimo = grupo_faixa['score'].min()
    pontos_de_corte.append(score_minimo)
 
# Ordenar do mais restritivo para o mais permissivo
pontos_de_corte = sorted(pontos_de_corte, reverse=True)
 
# Calcular as metricas para cada ponto de corte
resultados_corte = []
 
scores_reais = df_faixas['score'].values
classes_reais = df_faixas['classe_real'].values
total_linhas = len(classes_reais)
 
for corte in pontos_de_corte:
 
    predicoes = (scores_reais >= corte).astype(int)
 
    # Blocos da matriz de confusao
    tp = ((predicoes == 1) & (classes_reais == 1)).sum()
    tn = ((predicoes == 0) & (classes_reais == 0)).sum()
    fp = ((predicoes == 1) & (classes_reais == 0)).sum()
    fn = ((predicoes == 0) & (classes_reais == 1)).sum()
 
    acuracia = (tp + tn) / total_linhas
 
    precisao_evento = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_evento = tp / (tp + fn) if (tp + fn) > 0 else 0
 
    precisao_nao_evento = tn / (tn + fn) if (tn + fn) > 0 else 0
    recall_nao_evento = tn / (tn + fp) if (tn + fp) > 0 else 0
 
    qtd_predito_evento = (predicoes == 1).sum()
    pct_predito_evento = qtd_predito_evento / total_linhas
 
    resultados_corte.append({
        'threshold': round(corte, 4),
        'pct_classificado_evento': round(pct_predito_evento, 4),
        'acuracia': round(acuracia, 4),
        'precisao_evento': round(precisao_evento, 4),
        'recall_evento': round(recall_evento, 4),
        'precisao_nao_evento': round(precisao_nao_evento, 4),
        'recall_nao_evento': round(recall_nao_evento, 4),
    })
 
df_pontos_corte = pd.DataFrame(resultados_corte)
 
print("Analise de pontos de corte - base OOT")
print(f"Total de observacoes: {total_obs}")
print(f"Taxa media de eventos: {taxa_media:.4f}")
print()
print(df_pontos_corte.to_string(index=False))

### Registro do modelo no mlflow

In [ ]:
# Logar o experimento no MLflow
#
# O MLflow registra 3 coisas sobre o modelo:
#   1. Parametros (hiperparametros usados)
#   2. Metricas (resultados de performance)
#   3. Artefatos (o modelo treinado, que pode ser carregado depois)

import mlflow
import mlflow.sklearn

# Desabilitar log automatico para controlar manualmente
mlflow.sklearn.autolog(disable=True)

# Iniciar um "run" no MLflow
with mlflow.start_run(run_name="rf_final_13vars") as run:

    # 1. Logar os hiperparametros
    mlflow.log_params(params_finais)

    # 2. Logar as metricas de performance
    mlflow.log_metrics({
        'auc_oot': auc_final,
        'precisao_evento': prec_final,
        'recall_evento': rec_final,
        'f1_evento': f1_final,
        'lift_top5': lift_top5,
        'lift_top10': lift_top10,
    })

    # 3. Logar o modelo treinado como artefato
    mlflow.sklearn.log_model(
        modelo_final,
        name="modelo",
        input_example=X_treino.iloc[:3],
    )

    run_id = run.info.run_id

print("Experimento logado no MLflow com sucesso!")
print(f"  Run ID: {run_id}")
print(f"  Run Name: rf_final_13vars")

In [ ]:
# Registrar o modelo no Unity Catalog
#
# Isso permite:
#   - Versionar o modelo (v1, v2, v3...)
#   - Servir em producao via endpoint
#   - Compartilhar com outros times

from mlflow.models import infer_signature

# Configurar o MLflow para usar o Unity Catalog como registry
mlflow.set_registry_uri("databricks-uc")

# Nome do modelo no formato: catalogo.schema.nome_modelo
NOME_MODELO = "<catalogo>.<schema>.<nome_modelo>"

# Criar a signature (define o formato de entrada e saida do modelo)
# A saida deve ser um DataFrame com coluna nomeada para gerar ColSpec (campos com "name" e "type")
# Usar ndarray 1D gera TensorSpec sem "name", incompativel com a pipeline de validacao
output_df = pd.DataFrame({'prediction': modelo_final.predict_proba(X_teste)[:, 1]})

signature = infer_signature(
    model_input=X_teste,
    model_output=output_df,
)

print("Signature do modelo:")
print(signature)
print()
print("Output signature JSON:")
print(signature.outputs.to_json())

# Registrar com uma nova run dedicada ao registro
with mlflow.start_run(run_name="rf_final_13vars_registro") as run:

    # Logar parametros e metricas novamente (boa pratica para rastreabilidade)
    mlflow.log_params(params_finais)
    mlflow.log_metrics({
        'auc_oot': auc_final,
        'precisao_evento': prec_final,
        'recall_evento': rec_final,
        'f1_evento': f1_final,
    })

    # Logar e registrar o modelo no Unity Catalog
    model_info = mlflow.sklearn.log_model(
        modelo_final,
        name="modelo",
        signature=signature,
        input_example=X_teste.iloc[:5],
        registered_model_name=NOME_MODELO,
    )

print(f"  Nome: {NOME_MODELO}")
print(f"  URI: {model_info.model_uri}")

### Carregar o modelo registrado como Spark UDF (escoragem em lote)

```python
import mlflow
modelo_udf = mlflow.pyfunc.spark_udf(spark, "models:/<catalogo>.<schema>.<nome_modelo>@champion")
df_score = df_spark.withColumn("score", modelo_udf(*features))
```

# Painel de monitoramento

In [ ]:
# PSI POR FAIXA — base esperada (treino) × base atual (produção)
import numpy as np
import pandas as pd


def calcular_psi(esperado: pd.Series, atual: pd.Series, eps: float = 1e-4) -> pd.DataFrame:
    """PSI por faixa. ``esperado`` e ``atual`` são os rótulos de faixa de cada registro
    (ex.: ``optb.transform(x, metric="bins")`` com o optimal binning ajustado no treino)."""
    faixas = sorted(set(esperado.dropna()) | set(atual.dropna()), key=str)
    pe = esperado.value_counts(normalize=True).reindex(faixas, fill_value=0).clip(lower=eps)
    pa = atual.value_counts(normalize=True).reindex(faixas, fill_value=0).clip(lower=eps)
    tabela = pd.DataFrame({"faixa": faixas, "pct_esperado": pe.values, "pct_atual": pa.values})
    tabela["psi"] = (tabela["pct_atual"] - tabela["pct_esperado"]) * np.log(tabela["pct_atual"] / tabela["pct_esperado"])
    return tabela


# PARA UTILIZAR:
# optb = OptimalBinning(name="X1", dtype="numerical", solver="cp").fit(df_treino["X1"], df_treino["target"])
# tabela = calcular_psi(pd.Series(optb.transform(df_treino["X1"], metric="bins")),
#                       pd.Series(optb.transform(df_producao["X1"], metric="bins")))
# print(f"PSI total: {tabela['psi'].sum():.4f}")

Painel de monitoramento: compara o comportamento da base de **treino (esperado)** com a base de
**produção (atual)** ao longo das faixas de cada variável e do score.

O **PSI** (Population Stability Index) mede o quanto a distribuição de uma variável mudou entre duas bases:

* **Base esperada (referência):** o público com o qual o modelo foi desenvolvido.
* **Base atual (produção):** o público que está passando pelo modelo em um período (ex.: a safra do mês).

$$\text{PSI} = \sum_i (\%\,\text{atual}_i - \%\,\text{esperado}_i) \times \ln\frac{\%\,\text{atual}_i}{\%\,\text{esperado}_i}$$

| PSI | Leitura |
|---|---|
| < 0,10 | estável |
| 0,10 – 0,25 | atenção: investigar |
| > 0,25 | instável: o modelo pode não representar mais a população |

Se os novos clientes têm o mesmo perfil dos clientes do passado, o PSI fica próximo de zero. O PSI não precisa do
target, então a safra pode ser monitorada antes de se conhecer o desfecho.

> Versão de produção (testada): `src/credit_scoring/monitoring.py`; painel AI/BI em `src/dashboards/`.